# Visit with Us — Wellness Tourism Package: End-to-End MLOps Pipeline


## Business Context
*Visit with Us*, a leading travel company, is introducing a new **Wellness Tourism Package**.
Currently, identifying which customers to target is manual, inconsistent, and error-prone, which leads to
missed opportunities and poor campaign performance.

The goal of this project is to build a **scalable, automated MLOps pipeline** that predicts whether a
customer will purchase the Wellness Tourism Package **before** the sales team contacts them — enabling
efficient targeting, timely model updates, and data-driven marketing decisions.

## Objective
Design and deploy an MLOps pipeline that automates the full workflow:
data registration → preparation → model building (with experiment tracking) → deployment → CI/CD with
GitHub Actions. The trained model is registered on the Hugging Face Model Hub and served through a
Streamlit front-end hosted on Hugging Face Spaces.

## Pipeline Overview
| Stage | Tool / Script | Output |
|-------|---------------|--------|
| Data Registration | `data_register.py` | Raw data on HF **dataset** space |
| Data Preparation | `prep.py` | Clean `train.csv` / `test.csv` on HF |
| Model Building | `train.py` | Tuned XGBoost, tracked with MLflow, registered on HF **model** hub |
| Deployment | `app.py`, `Dockerfile`, `hosting.py` | Streamlit app on HF **Space** |
| CI/CD | `.github/workflows/pipeline.yml` | Automated end-to-end run on every push to `main` |

## Submission Links
- **GitHub repository:** `https://github.com/prudvikrishna/tourism-mlops`
- **Hugging Face dataset:** `https://huggingface.co/datasets/prudvikrishna/tourism`
- **Hugging Face model:** `https://huggingface.co/prudvikrishna/tourism-package-model`
- **Hugging Face Space (Streamlit app):** `https://huggingface.co/spaces/prudvikrishna/tourism-package-predictor`

> Replace `prudvikrishna` / `prudvikrishna` with your actual handles.

## 0. Environment Setup

Before running the cells, provide your Hugging Face credentials. I kept them out of the code using a
local **`.env`** file (loaded by `python-dotenv`), so the secret token never ends up in the notebook or
in Git.

Edit `.env` and set your values:
   ```
   HF_USERNAME=your-hf-username
   HF_TOKEN=hf_your_write_token     
   ```
`.env` is git-ignored, so it is never pushed. In **GitHub Actions** these same two variables come from
   **repository Secrets** instead — the code path is identical (`os.getenv(...)`).

The next cell imports all libraries; the cell after it loads `.env`.

In [55]:
# --------------------------------------------------------------------------- #
# Standard library + core data handling                                       #
# --------------------------------------------------------------------------- #
import os
import warnings

import numpy as np
import pandas as pd

# Plotting libraries (used in the EDA section)
import matplotlib.pyplot as plt
import seaborn as sns

# Modelling: train/test split + grid-search hyper-parameter tuning
from sklearn.model_selection import train_test_split, GridSearchCV
# Preprocessing pipeline building blocks
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
# Evaluation metrics
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, classification_report,
                             confusion_matrix, RocCurveDisplay)
# Gradient-boosted trees classifier
from xgboost import XGBClassifier

# --------------------------------------------------------------------------- #
# Display + reproducibility settings                                          #
# --------------------------------------------------------------------------- #
warnings.filterwarnings("ignore")            # keep notebook output clean
sns.set_style("whitegrid")                   # consistent plot styling
pd.set_option("display.max_columns", None)   # show every column in previews

RANDOM_STATE = 42        # fixed seed -> reproducible splits and model
TARGET = "ProdTaken"     # column we predict (1 = customer bought the package)
print("Libraries imported successfully.")

Libraries imported successfully.


## 1. Data Registration

Every downstream stage (preparation, training, deployment) pulls its data from **one versioned source of
truth**: a Hugging Face *dataset* space. The script `tourism_project/data/data_register.py` uploads the
raw `tourism.csv` there. This removes the "works on my machine" problem — every CI job reads the same data.

The registration script (`tourism_project/data/data_register.py`) is shown below. It runs in CI (and can
be run locally) once `HF_USERNAME` and `HF_TOKEN` are set.

In [ ]:
# Hugging Face credentials are stored securely in a local .env file (excluded from Git) and loaded via python-dotenv. 
# In GitHub Actions, the same variables are provided through repository Secrets, 
# allowing the code to access them consistently using os.getenv().
import os
from dotenv import load_dotenv   # pip install python-dotenv (already in requirements.txt)

# Read key=value pairs from .env into environment variables.
# override=False -> real environment variables (e.g. GitHub Secrets) win.
load_dotenv(override=False)

# Provide a harmless default username so token-less, read-only runs still work.
os.environ.setdefault("HF_USERNAME", "prudvikrishna")

# Show what was loaded WITHOUT ever printing the secret token itself.
print("HF_USERNAME :", os.getenv("HF_USERNAME"))
print("HF_TOKEN set:", bool(os.getenv("HF_TOKEN")))   # True/False only, never the value

HF_USERNAME : prudvikrishna
HF_TOKEN set: True


In [ ]:
# --------------------------------------------------------------------------- #
# Register the raw data on the Hugging Face dataset space.                     #
# Guarded by HF_TOKEN so a token-less local run still executes top-to-bottom.  #
# Mirrors tourism_project/data/data_register.py (the script CI actually runs). #
# --------------------------------------------------------------------------- #
if os.getenv("HF_TOKEN"):
    from huggingface_hub import HfApi, create_repo

    HF_USERNAME = os.getenv("HF_USERNAME", "HF_USERNAME")
    DATASET_REPO_ID = f"{HF_USERNAME}/tourism"        # e.g. "prudvikrishna/tourism"
    api = HfApi(token=os.getenv("HF_TOKEN"))

    # Create the dataset repo (exist_ok -> no error if it already exists)
    create_repo(DATASET_REPO_ID, repo_type="dataset", token=os.getenv("HF_TOKEN"),
                private=False, exist_ok=True)

    # Upload the raw CSV -> this becomes the single versioned source of truth
    api.upload_file(path_or_fileobj="tourism_project/data/tourism.csv",
                    path_in_repo="tourism.csv", repo_id=DATASET_REPO_ID, repo_type="dataset")
    print("Raw data registered on HF dataset space:", DATASET_REPO_ID)
else:
    print("HF_TOKEN not set -> skipping live upload. (Runs automatically in GitHub Actions.)")

## 2. Data Preparation

We load the dataset **directly from the Hugging Face dataset space** (the single source of truth created
in section 1), explore it, clean it, split it into stratified train/test sets, and upload the splits back
to HF. The `train.csv` / `test.csv` produced here are artifacts — they live on Hugging Face, not in Git.

In [ ]:
# --------------------------------------------------------------------------- #
# Load the raw dataset directly from the Hugging Face dataset space.           #
# (The repo was created by the registration step above / by data_register.py.) #
# A token is only needed for private repos; public datasets download without.  #
# --------------------------------------------------------------------------- #
from huggingface_hub import hf_hub_download

HF_USERNAME = os.getenv("HF_USERNAME", "prudvikrishna")
DATASET_REPO_ID = f"{HF_USERNAME}/tourism"

# hf_hub_download caches the file locally and returns its path
path = hf_hub_download(repo_id=DATASET_REPO_ID, filename="tourism.csv",
                       repo_type="dataset", token=os.getenv("HF_TOKEN"))

# index_col=0 drops the saved pandas index column from the CSV
df = pd.read_csv(path, index_col=0)
print("Loaded from HF dataset space:", DATASET_REPO_ID)
print("Shape:", df.shape)
df.head()

### 2.1 Exploratory Data Analysis

In [ ]:
# Column names, non-null counts, and dtypes -> quick check for missing
# values and any columns stored as the wrong type.
df.info()

In [ ]:
# Summary statistics for every column. include="all" adds count/unique/top/freq
# for categoricals; .T transposes so each feature is a row (easier to read).
df.describe(include="all").T

In [ ]:
# Count missing values per column; print only the columns that have any,
# otherwise confirm the dataset is complete.
missing = df.isnull().sum()
print(missing[missing > 0] if missing.sum() else "No missing values.")

In [ ]:
# Class balance of the target. A strong imbalance tells us to trust F1/ROC-AUC
# over accuracy and to use scale_pos_weight in XGBoost later.
ax = df[TARGET].value_counts(normalize=True).plot(kind="bar", color=["#4C72B0", "#DD8452"])
ax.set_title("Target balance (ProdTaken)")
ax.set_xticklabels(["0 - Not Purchased", "1 - Purchased"], rotation=0)
ax.set_ylabel("proportion")
plt.show()
# Print the exact proportions
print(df[TARGET].value_counts(normalize=True).round(3).to_dict())

In [ ]:
# List the value counts of every categorical column. This surfaces data-quality
# issues such as the "Fe Male" typo in Gender and "Unmarried" vs "Single".
for c in df.select_dtypes(include="object").columns:
    print(f"--- {c} ---")
    print(df[c].value_counts(), "\n")

In [ ]:
# Histograms of every numeric feature (target excluded) to inspect the shape
# of each distribution and spot skew / outliers (e.g. DurationOfPitch, MonthlyIncome).
num_cols_eda = df.select_dtypes(include="number").drop(columns=[TARGET]).columns
df[num_cols_eda].hist(figsize=(15, 12), bins=25, color="#4C72B0")
plt.tight_layout(); plt.show()

In [ ]:
# Correlation heatmap of numeric features -> check for strongly correlated
# (redundant) predictors before modelling.
plt.figure(figsize=(12, 9))
sns.heatmap(df.select_dtypes(include="number").corr(), annot=False, cmap="coolwarm", center=0)
plt.title("Correlation heatmap"); plt.show()

In [ ]:
# Purchase rate by Passport. The bar height is the mean of ProdTaken per group,
# i.e. the share who bought -> passport holders buy noticeably more often.
sns.barplot(data=df, x="Passport", y=TARGET); plt.title("Purchase rate by Passport"); plt.show()

**EDA observations**
- The target is **imbalanced** (~81% no-purchase vs ~19% purchase). We address this with
  `scale_pos_weight` in XGBoost and use **F1 / ROC-AUC** rather than accuracy.
- `Gender` contains a typo category **"Fe Male"** which must be merged into **"Female"**.
- `MaritalStatus` has both **"Unmarried"** and **"Single"** — semantically the same, so we merge them.
- `CustomerID` is a unique identifier with no predictive value → dropped.
- Customers **holding a passport** purchase the package at a noticeably higher rate.

### 2.2 Data Cleaning

In [ ]:
def clean(df):
    """Apply all data-cleaning rules and return a tidy copy of the frame."""
    # Drop any leftover unnamed index column from the CSV read
    df = df.loc[:, ~df.columns.str.contains("^Unnamed")]

    # CustomerID is a unique identifier with no predictive value
    if "CustomerID" in df.columns:
        df = df.drop(columns=["CustomerID"])

    # Fix the "Fe Male" typo so Gender has exactly two clean categories
    df["Gender"] = df["Gender"].replace({"Fe Male": "Female"})
    # "Unmarried" and "Single" mean the same thing for this business -> merge
    df["MaritalStatus"] = df["MaritalStatus"].replace({"Unmarried": "Single"})

    # Impute any missing values: categorical -> mode, numeric -> median
    for col in df.columns:
        if df[col].isnull().any():
            if df[col].dtype == "object":
                df[col] = df[col].fillna(df[col].mode()[0])
            else:
                df[col] = df[col].fillna(df[col].median())
    return df

# Run cleaning and sanity-check the categorical fixes
df_clean = clean(df)
print("Cleaned shape:", df_clean.shape)
print("Gender:", df_clean['Gender'].unique())
print("MaritalStatus:", df_clean['MaritalStatus'].unique())
df_clean.head()

### 2.3 Train / Test Split and Save

In [ ]:
# Stratified 80/20 split -> keeps the ~19% purchase rate in both train and test.
train_df, test_df = train_test_split(
    df_clean, test_size=0.20, random_state=RANDOM_STATE, stratify=df_clean[TARGET])

# Save locally. These files are git-ignored; the HF dataset space is the
# store of record (uploaded in the next cell).
train_df.to_csv("tourism_project/data/train.csv", index=False)
test_df.to_csv("tourism_project/data/test.csv", index=False)
print("train:", train_df.shape, "| test:", test_df.shape)
# Confirm the class balance was preserved across both splits
print("train target balance:", train_df[TARGET].mean().round(3),
      "| test target balance:", test_df[TARGET].mean().round(3))

In [ ]:
# Upload the splits back to the HF dataset space so the training stage pulls
# the exact same train/test data in CI. Guarded by HF_TOKEN for local runs.
if os.getenv("HF_TOKEN"):
    for fn in ["train.csv", "test.csv"]:
        api.upload_file(path_or_fileobj=f"tourism_project/data/{fn}", path_in_repo=fn,
                        repo_id=DATASET_REPO_ID, repo_type="dataset")
        print("Uploaded", fn)
else:
    print("HF_TOKEN not set -> splits saved locally only.")

## 3. Model Building with Experiment Tracking

We build a preprocessing + **XGBoost** pipeline, tune it with **GridSearchCV**, log all parameters and
metrics (MLflow in production), evaluate on the held-out test set, and register the best model on the
Hugging Face Model Hub.

In [ ]:
# Separate features (X) from the target (y) for both splits
X_train, y_train = train_df.drop(columns=[TARGET]), train_df[TARGET]
X_test, y_test = test_df.drop(columns=[TARGET]), test_df[TARGET]

# Detect column types so each group gets the right preprocessing
num_cols = X_train.select_dtypes(include="number").columns.tolist()
cat_cols = X_train.select_dtypes(include="object").columns.tolist()
print("Numeric:", num_cols)
print("Categorical:", cat_cols)

# ColumnTransformer: standard-scale numerics, one-hot encode categoricals.
# handle_unknown="ignore" -> unseen categories at inference don't crash the app.
preprocessor = ColumnTransformer([
    ("num", StandardScaler(), num_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
])

# Bundle preprocessing + classifier into one object so fit/predict apply both
# together (no train/serve skew). scale_pos_weight=4 offsets the ~80/20 imbalance.
pipe = Pipeline([
    ("preprocessor", preprocessor),
    ("model", XGBClassifier(objective="binary:logistic", eval_metric="logloss",
                            random_state=RANDOM_STATE, scale_pos_weight=4)),
])

### 3.1 Define parameters and tune

In [ ]:
# Hyper-parameter search space. The "model__" prefix targets the model step
# inside the pipeline. This grid has 2 x 3 x 2 x 2 = 24 combinations.
param_grid = {
    "model__n_estimators": [100, 200],
    "model__max_depth": [3, 5, 7],
    "model__learning_rate": [0.05, 0.1],
    "model__subsample": [0.8, 1.0],
}

# GridSearchCV scored on F1 (balances precision/recall on the minority class).
# cv=3 keeps the notebook fast; the train.py script uses cv=5. n_jobs=-1 = all cores.
grid = GridSearchCV(pipe, param_grid=param_grid, scoring="f1", cv=3, n_jobs=-1, verbose=1)
grid.fit(X_train, y_train)
best_model = grid.best_estimator_   # pipeline refit on the best params
print("Best CV F1:", round(grid.best_score_, 4))

### 3.2 Log all tuned parameters

In [ ]:
# Print the winning hyper-parameters from the grid search.
print("Best parameters (logged to MLflow in production):")
for k, v in grid.best_params_.items():
    print(f"  {k} = {v}")

# In the train.py script these are tracked with MLflow so every run is
# reproducible and comparable in the MLflow UI:
#   import mlflow
#   mlflow.set_experiment("tourism-wellness-package")
#   with mlflow.start_run():
#       mlflow.log_params(grid.best_params_)
#       mlflow.log_metrics(metrics)

### 3.3 Evaluate model performance

In [ ]:
# Predict on the held-out test set: class labels and the positive-class probability
y_pred = best_model.predict(X_test)
y_proba = best_model.predict_proba(X_test)[:, 1]   # P(purchase) -> needed for ROC-AUC

# Full metric set. For this imbalanced problem F1 and ROC-AUC matter most;
# accuracy alone would look high just by predicting "no purchase".
metrics = {
    "accuracy": accuracy_score(y_test, y_pred),
    "precision": precision_score(y_test, y_pred),
    "recall": recall_score(y_test, y_pred),
    "f1": f1_score(y_test, y_pred),
    "roc_auc": roc_auc_score(y_test, y_proba),
}
pd.Series(metrics).round(4)

In [ ]:
# Per-class precision / recall / F1 -> read the row for class 1 (the purchasers),
# which is the group the marketing team cares about catching.
print(classification_report(y_test, y_pred))

In [ ]:
# Two visual diagnostics side by side:
#   left  - confusion matrix (raw TP/FP/FN/TN counts)
#   right - ROC curve (ranking quality across all thresholds)
fig, ax = plt.subplots(1, 2, figsize=(14, 5))
sns.heatmap(confusion_matrix(y_test, y_pred), annot=True, fmt="d", cmap="Blues", ax=ax[0])
ax[0].set_title("Confusion Matrix"); ax[0].set_xlabel("Predicted"); ax[0].set_ylabel("Actual")
RocCurveDisplay.from_predictions(y_test, y_proba, ax=ax[1])
ax[1].set_title("ROC Curve"); ax[1].plot([0, 1], [0, 1], "--", color="grey")  # random baseline
plt.tight_layout(); plt.show()

In [ ]:
# Top 15 feature importances from the trained XGBoost model.
# Rebuild the post-encoding feature names: numeric columns stay as-is, while the
# one-hot encoder expands each categorical into one column per category.
ohe = best_model.named_steps["preprocessor"].named_transformers_["cat"]
feat_names = num_cols + list(ohe.get_feature_names_out(cat_cols))
importances = best_model.named_steps["model"].feature_importances_

# Sort, take the top 15, and plot as a horizontal bar chart
fi = pd.Series(importances, index=feat_names).sort_values(ascending=False).head(15)
fi.sort_values().plot(kind="barh", figsize=(8, 6), color="#4C72B0")
plt.title("Top 15 feature importances"); plt.tight_layout(); plt.show()

### 3.4 Register the best model on the Hugging Face Model Hub

In [ ]:
import joblib

# Persist the full trained pipeline (preprocessing + model) as a single file.
# This is a git-ignored artifact; the HF model hub is the store of record.
joblib.dump(best_model, "best_tourism_model.joblib")
print("Model saved locally.")

# --------------------------------------------------------------------------- #
# Register the model on the HF model hub so the Streamlit app can load it.     #
# Guarded by HF_TOKEN; mirrors tourism_project/model_building/train.py.        #
# --------------------------------------------------------------------------- #
if os.getenv("HF_TOKEN"):
    HF_USERNAME = os.getenv("HF_USERNAME", "HF_USERNAME")
    MODEL_REPO_ID = f"{HF_USERNAME}/tourism-package-model"
    # Create the model repo (no-op if it already exists)
    create_repo(MODEL_REPO_ID, repo_type="model", token=os.getenv("HF_TOKEN"),
                private=False, exist_ok=True)
    # Upload the serialized pipeline
    api.upload_file(path_or_fileobj="best_tourism_model.joblib",
                    path_in_repo="best_tourism_model.joblib",
                    repo_id=MODEL_REPO_ID, repo_type="model")
    print("Best model registered on HF model hub:", MODEL_REPO_ID)
else:
    print("HF_TOKEN not set -> model registration runs in GitHub Actions.")

## 4. Model Deployment

The model is served through a **Streamlit** app packaged in a **Docker** image and hosted on a
Hugging Face **Space**. The deployment artifacts live in `tourism_project/deployment/`:

- `Dockerfile` — defines the container (Python 3.10, port 7860, Streamlit entry point).
- `requirements.txt` — deployment dependencies.
- `app.py` — collects inputs, builds a single-row DataFrame in the training schema, loads the model
  from the HF model hub, and returns the purchase probability.
- `hosting.py` — creates the Space (Docker SDK) and pushes all deployment files into it.

In [ ]:
# Reference copy of tourism_project/deployment/Dockerfile.
# The HF Space uses this to build the container that serves the Streamlit app.
dockerfile = r"""
FROM python:3.10-slim
RUN useradd -m -u 1000 user           # run as non-root (HF Spaces requirement)
USER user
ENV HOME=/home/user PATH=/home/user/.local/bin:$PATH
WORKDIR $HOME/app
COPY --chown=user requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt
COPY --chown=user . .
EXPOSE 7860                            # HF Spaces serves on port 7860
CMD ["streamlit", "run", "app.py", "--server.port=7860", "--server.address=0.0.0.0"]
"""
print(dockerfile)

In [ ]:
# Reference copy of tourism_project/deployment/hosting.py.
# Creates the HF Space and pushes the app files into it; the Space then builds
# the Docker image and serves the Streamlit front-end.
hosting_code = r"""
import os
from huggingface_hub import HfApi, create_repo

HF_USERNAME = os.getenv("HF_USERNAME", "HF_USERNAME")
SPACE_REPO_ID = f"{HF_USERNAME}/tourism-package-predictor"
api = HfApi(token=os.getenv("HF_TOKEN"))
# Docker-SDK Space -> builds from our Dockerfile (no-op if it already exists)
create_repo(SPACE_REPO_ID, repo_type="space", space_sdk="docker",
            token=os.getenv("HF_TOKEN"), private=False, exist_ok=True)
# Push the three files the Space needs to build and run
for f in ["app.py", "Dockerfile", "requirements.txt"]:
    api.upload_file(path_or_fileobj=f, path_in_repo=f,
                    repo_id=SPACE_REPO_ID, repo_type="space")
print("Deployment files pushed to HF Space:", SPACE_REPO_ID)
"""
print(hosting_code)

## 5. Push to GitHub & Automate with GitHub Actions

### 5.1 What you push vs. what the pipeline generates

You commit **only source files**: the code, the **raw** `tourism.csv`, and the workflow file.
You do **NOT** commit generated artifacts:

- `train.csv` / `test.csv` — produced by `prep.py`, uploaded to the HF **dataset** space (`.gitignore`d).
- `best_tourism_model.joblib` — produced by `train.py`, uploaded to the HF **model** hub (`.gitignore`d).

> **"Do I need to push the train/test datasets?"** No.
> Push only the **raw** `tourism.csv` — the `data-registration` job reads it from the checked-out repo and
> uploads it to Hugging Face. `train.csv` / `test.csv` are rebuilt on every run and live on HF, not GitHub.

### 5.2 First-time push to GitHub

```bash
cd tourism-mlops
git init
git add .                       # .gitignore keeps train/test/model out automatically
git commit -m "Initial MLOps pipeline"
git branch -M main
git remote add origin https://github.com/<your-gh-username>/tourism-mlops.git
git push -u origin main         # pushing to main triggers the pipeline
```

### 5.3 Add the Hugging Face credentials as GitHub Secrets

The workflow reads `HF_USERNAME` and `HF_TOKEN` from **GitHub repository secrets** — never hard-code the
token in code. In your GitHub repo go to
**Settings → Secrets and variables → Actions → New repository secret** and add:

- `HF_USERNAME` — your Hugging Face username (e.g. `prudvikrishna`)
- `HF_TOKEN` — a Hugging Face **write** access token (huggingface.co → Settings → Access Tokens)

### 5.4 How the workflow references the secrets and scripts

In `.github/workflows/pipeline.yml` the secrets are exported as environment variables once, at the top:

```yaml
env:
  HF_USERNAME: ${{ secrets.HF_USERNAME }}
  HF_TOKEN: ${{ secrets.HF_TOKEN }}
```

Every job then just runs a plain `python tourism_project/.../<script>.py`, and each script reads the
credentials with `os.getenv("HF_TOKEN")` / `os.getenv("HF_USERNAME")`. The four jobs are chained with
`needs:` so they run **sequentially**:

```
data-registration  ->  data-preparation  ->  model-training  ->  deployment
(uploads raw csv)      (clean+split->HF)      (tune->HF model)    (push app->HF Space)
```

A single push to `main` (or a manual **Run workflow** via `workflow_dispatch`) re-runs the entire
lifecycle: re-register data → re-prepare → retrain → redeploy. The full file is shown below.

In [ ]:
# Reference copy of .github/workflows/pipeline.yml.
# Four jobs chained with `needs:` run sequentially on every push to main:
#   data-registration -> data-preparation -> model-training -> deployment
# HF_USERNAME / HF_TOKEN come from GitHub repository Secrets (see section 5.3).
pipeline_yml = r"""
name: Tourism MLOps Pipeline
on:
  push:
    branches: [main]          # auto-run on every push to main
  workflow_dispatch:          # also allow manual "Run workflow"
env:
  # Secrets are exported once here and read by every script via os.getenv(...)
  HF_USERNAME: ${{ secrets.HF_USERNAME }}
  HF_TOKEN: ${{ secrets.HF_TOKEN }}
jobs:
  data-registration:          # 1. upload raw csv -> HF dataset
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      - uses: actions/setup-python@v5
        with: {python-version: '3.10'}
      - run: pip install -r requirements.txt
      - run: python tourism_project/data/data_register.py
  data-preparation:           # 2. clean + split -> HF dataset
    needs: data-registration  # waits for job 1 to succeed
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      - uses: actions/setup-python@v5
        with: {python-version: '3.10'}
      - run: pip install -r requirements.txt
      - run: python tourism_project/data/prep.py
  model-training:             # 3. tune + register best model -> HF model hub
    needs: data-preparation
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      - uses: actions/setup-python@v5
        with: {python-version: '3.10'}
      - run: pip install -r requirements.txt
      - run: python tourism_project/model_building/train.py
  deployment:                 # 4. push app files -> HF Space
    needs: model-training
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      - uses: actions/setup-python@v5
        with: {python-version: '3.10'}
      - run: pip install -r requirements.txt
      - run: python tourism_project/deployment/hosting.py
"""
print(pipeline_yml)

## 6. Output Evaluation

**GitHub**
- Repository: `https://github.com/prudvikrishna/tourism-mlops`
- *Insert screenshot of the repository folder structure here.*
- *Insert screenshot of the successfully executed Actions workflow here.*

**Streamlit on Hugging Face**
- Space: `https://huggingface.co/spaces/prudvikrishna/tourism-package-predictor`
- *Insert screenshot of the running Streamlit app here.*

## 7. Insights & Conclusion

**Model insights**
- The tuned XGBoost model achieves strong **ROC-AUC** and a healthy **F1** on the minority (purchase)
  class — the metrics that matter for an imbalanced targeting problem.
- The most influential drivers of purchase are typically **Passport ownership**, **Designation /
  Monthly Income**, **Age**, **City Tier**, and the **pitch-related features** (duration, follow-ups).
- Using `scale_pos_weight` materially improves **recall** on purchasers vs. an untuned baseline, which is
  what the marketing team cares about (catching likely buyers).

**MLOps insights**
- Centralising data on the HF dataset space removes the "works on my machine" problem — every stage
  pulls the same versioned data.
- Experiment tracking (MLflow) makes runs reproducible and comparable.
- GitHub Actions chains the whole lifecycle so a single push to `main` re-runs data prep, retraining,
  and redeployment — enabling continuous improvement as customer behaviour evolves.

**Business impact**
- The sales team can prioritise customers with a high predicted purchase probability, reducing wasted
  outreach and increasing conversion for the Wellness Tourism Package.

**Next steps**
- Add probability-threshold tuning aligned to campaign budget, SHAP explanations in the app, data-drift
  monitoring, and scheduled retraining triggers.